# Búsqueda ternaria · laboratorio ejecutable

Este notebook conserva el código, los controles y la animación. La explicación, las ecuaciones y la interpretación de resultados se encuentran en el complemento digital:

<div style="margin:1.25rem 0 1.5rem;padding:18px 20px;border:1px solid #bdb9b2;border-left:4px solid #8b4b32;background:#fbfaf7;color:#242321;font-family:Arial,Helvetica,sans-serif;">
  <div style="margin-bottom:6px;color:#8b4b32;font-size:11px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;">Complemento digital</div>
  <div style="margin-bottom:14px;font-size:15px;line-height:1.55;">Consulta la explicación, las ecuaciones y la interpretación de resultados en el sitio de la obra.</div>
  <a class="notebook-pages-button" href="https://notas-a-mano-serie-de-libros.github.io/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/capitulos/capitulo-7/6-busqueda-ternaria/" target="_blank" rel="noopener noreferrer" aria-label="Leer la explicación completa en GitHub Pages; abre una pestaña nueva">
    <img src="https://img.shields.io/badge/LEER_LA_EXPLICACI%C3%93N_COMPLETA_EN_GITHUB_PAGES-33312e?style=for-the-badge&logo=github&logoColor=white" alt="Leer la explicación completa en GitHub Pages" width="430" />
  </a>
</div>

Ejecuta las celdas en orden y utiliza los controles de la simulación. Al finalizar, vuelve a Pages para contrastar los resultados con el análisis teórico.


In [ ]:
#@title Ejecutar simulación de búsqueda ternaria { display-mode: "form" }
from pathlib import Path
import urllib.request

SIMULATION_NAME = "ternaria"
BOOTSTRAP_URL = "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/main/capitulo7/runtime/colab_bootstrap.py"
BOOTSTRAP_CANDIDATES = (
    Path("capitulo7/runtime/colab_bootstrap.py"),
    Path("runtime/colab_bootstrap.py"),
    Path("colab_bootstrap.py"),
)

bootstrap = next((path for path in BOOTSTRAP_CANDIDATES if path.exists()), None)
bootstrap_code = (
    bootstrap.read_text(encoding="utf-8")
    if bootstrap
    else urllib.request.urlopen(BOOTSTRAP_URL).read().decode("utf-8")
)
exec(bootstrap_code)


In [ ]:
#@title Eficiencia por tamaño de arreglo { display-mode: "form" }
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import urllib.request

RAW_BASE_URL = "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/3_notas-a-mano-sobre-analisis-de-complejidad-computacional/main/core/search"
CHART_CANDIDATES = (
    Path("../../core/search/busquedas_chart.py"),
    Path("core/search/busquedas_chart.py"),
    Path("busquedas_chart.py"),
)
chart_path = next((path for path in CHART_CANDIDATES if path.exists()), None)
if chart_path is None:
    chart_path = Path("busquedas_chart.py")
    chart_path.write_text(urllib.request.urlopen(f"{RAW_BASE_URL}/busquedas_chart.py").read().decode("utf-8"), encoding="utf-8")
    Path("search_metrics.py").write_text(urllib.request.urlopen(f"{RAW_BASE_URL}/search_metrics.py").read().decode("utf-8"), encoding="utf-8")

spec = spec_from_file_location("cap7_busquedas_chart_runtime", chart_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f"No se pudo cargar {chart_path}")

chart_module = module_from_spec(spec)
spec.loader.exec_module(chart_module)
chart_module.run_single_chart("Ternaria")


In [ ]:
#@title Comparación asintótica: log₂(n) vs log₃(n) { display-mode: "form" }
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import math

matplotlib.rcParams.update({"figure.dpi": 500, "savefig.dpi": 500})

n = np.geomspace(2, 1e7, 600)
log2_n     = np.log2(n)
log3_n     = np.log(n) / np.log(3)
two_log3_n = 2 * log3_n

# ── Normalización: dividir por el valor en n₀ = 10 ──────────────────────────
n0        = 10.0
log2_norm = log2_n     / math.log2(n0)
log3_norm = log3_n     / (math.log(n0) / math.log(3))

fmt = mticker.FuncFormatter(
    lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6
    else f"{int(x/1e3)}k" if x >= 1e3
    else str(int(x))
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Panel izquierdo: valores absolutos ──────────────────────────────────────
C_BIN = "#1565C0"
C_TER = "#6A1B9A"

ax1.plot(n, log2_n,     color=C_BIN, linewidth=2.2,
         label=r"$\log_2 n$  (búsqueda binaria)")
ax1.plot(n, two_log3_n, color=C_TER, linewidth=2.2,
         label=r"$2\,\log_3 n$  (búsqueda ternaria)")
ax1.plot(n, log3_n,     color=C_TER, linewidth=1.4, linestyle="--", alpha=0.6,
         label=r"$\log_3 n$  (sin factor 2)")

ax1.set_xscale("log")
ax1.set_xlabel("n", fontsize=13)
ax1.set_ylabel("Valor de la función", fontsize=13)
ax1.set_title("Diferencia en valores absolutos", fontsize=13)
ax1.legend(fontsize=10, loc="upper left")
ax1.grid(True)
ax1.xaxis.set_major_formatter(fmt)

# Anotación del factor constante
mid_idx = len(n) // 2
y_log2  = log2_n[mid_idx]
y_log3  = log3_n[mid_idx]
ax1.annotate(
    f"factor $\\log_2 3 \\approx {math.log2(3):.3f}$",
    xy=(n[mid_idx], (y_log2 + y_log3) / 2),
    xytext=(n[mid_idx] * 3, (y_log2 + y_log3) / 2 + 1.5),
    fontsize=9, color="#555555",
    arrowprops=dict(arrowstyle="-", color="#aaaaaa", lw=0.8),
)

# ── Panel derecho: funciones normalizadas ────────────────────────────────────
ax2.plot(n, log2_norm, color=C_BIN, linewidth=2.5,
         label=r"$\log_2 n \;/\; \log_2 n_0$", zorder=3)
ax2.plot(n, log3_norm, color=C_TER, linewidth=1.8, linestyle="--", alpha=0.85,
         label=r"$\log_3 n \;/\; \log_3 n_0$", zorder=4)

ax2.set_xscale("log")
ax2.set_xlabel("n", fontsize=13)
ax2.set_ylabel(r"Valor normalizado  $(f(n)/f(n_0),\ n_0=10)$", fontsize=12)
ax2.set_title(
    "Comportamiento asintótico — curvas normalizadas\n"
    r"(superposición perfecta $\Rightarrow$ misma clase $\Theta$)",
    fontsize=12,
)
ax2.legend(fontsize=10, loc="upper left")
ax2.grid(True)
ax2.xaxis.set_major_formatter(fmt)

plt.suptitle(
    r"$\log_2 n$ y $\log_3 n$: diferencia concreta vs. equivalencia asintótica",
    fontsize=14,
)
plt.tight_layout()
plt.show()
